### Imports

In [2]:
#Importações e Passo a passo

#libs python
import os
from pathlib import Path
import sys
from dotenv import load_dotenv
import time

from google import genai #modelo
from google.genai import types #configurações system_intructions, temperature.
from google.genai.errors import APIError # para tratar erros
from langchain_community.document_loaders import PyPDFLoader #Loder para ler os PDFs.
from langchain_text_splitters import RecursiveCharacterTextSplitter #Dividir os PDFs em chuncks menores.
from langchain_community.vectorstores import FAISS #Armazenar os embeddings em um banco de dados vetorial.
from langchain_core.prompts import ChatPromptTemplate #Conversar com o documento
from langchain_core.output_parsers import SimpleJsonOutputParser #Output parseado

C:\Users\annab\AppData\Local\Temp\ipykernel_2456\3465956236.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader #Loder para ler os PDFs.


### Configuração de ambientes e modulos internos

In [3]:
#Configuração de Paths e modelo

load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
CLIENT = genai.Client(api_key=API_KEY)
EMBEDDING_MODEL = "gemini-embedding-001"

project_root = Path("../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.ingestion.candidates import CANDIDATOS_MAP #Importa o mapeamento de candidatos

'''modelo = CLIENT.models.generate_content(
    model="gemini-3.8-flash",
    contents="prompt"
)'''

'modelo = CLIENT.models.generate_content(\n    model="gemini-3.8-flash",\n    contents="prompt"\n)'

### Caminhos de PDFs, leitura e estrutura

In [4]:
#2. Definir lista com caminhos dos pdfs
def carregar_e_enriquecer_pdf(pdf_path: Path, candidatos_map: dict) -> list:
    file_id = pdf_path.stem # cria id para  arquivo removendo o .pdf

    #map de metadados
    info_candidato = CANDIDATOS_MAP.get(file_id, { 
        "nome_candidato": f"CandidatoNotFound{file_id}",
        "nome_vice": "N/A",
        "nome_partido": "N/A",
        "sigla_partido": "N/A"
    })

    docs = PyPDFLoader(str(pdf_path)).load() #loader instanciado e loaded

    #injetar meta dados
    for doc in docs:
        doc.metadata.update({
            "nome_candidato": info_candidato["nome_candidato"],
            "nome_vice": info_candidato["nome_vice"],
            "nome_partido": info_candidato["nome_partido"],
            "sigla_partido": info_candidato["sigla_partido"],
            "arquivo_origem": pdf_path.name
        })

    return docs

print("inicia leitura de pdf")
data_dir = Path("../data/documentos_2026") 
pdf_files = list(data_dir.glob("*.pdf"))
all_documents = []

for pdf in pdf_files:
    all_documents.extend(carregar_e_enriquecer_pdf(pdf, CANDIDATOS_MAP))

print(f"Total de {len(all_documents)} páginas em {len(pdf_files)} arquivos")


inicia leitura de pdf
Total de 160 páginas em 2 arquivos


### Transforma texto dos pdfs em chunks

In [5]:
# 3. Divide texto em chunks
print("inicia divisão de chunks")
chunks = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True
).split_documents(all_documents)
print(f"Total de {len(chunks)} chunks")

inicia divisão de chunks
Total de 461 chunks


### Verificar modelos disponíveis

In [24]:
import os
from dotenv import load_dotenv
from google import genai

# 1. Carregar variáveis de ambiente (.env)
load_dotenv()

# 2. Instanciar o cliente genai oficial
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("Buscando modelos disponíveis para a sua API Key...\n")

# 3. Listar todos os modelos suportados pela sua chave
models = client.models.list()

print(f"{'NOME DO MODELO':<45} | {'MÉTODOS SUPORTADOS'}")
print("-" * 80)

for m in models:
    # Filtra modelos que aceitam geração de conteúdo ou embeddings
    methods = getattr(m, 'supported_generation_methods', [])
    print(f"{m.name:<45} | {methods}")

Buscando modelos disponíveis para a sua API Key...

NOME DO MODELO                                | MÉTODOS SUPORTADOS
--------------------------------------------------------------------------------
models/gemini-2.5-flash                       | []
models/gemini-2.5-pro                         | []
models/gemini-2.5-flash-preview-tts           | []
models/gemini-2.5-pro-preview-tts             | []
models/gemma-4-26b-a4b-it                     | []
models/gemma-4-31b-it                         | []
models/gemini-flash-latest                    | []
models/gemini-flash-lite-latest               | []
models/gemini-pro-latest                      | []
models/gemini-2.5-flash-lite                  | []
models/gemini-2.5-flash-image                 | []
models/gemini-3-flash-preview                 | []
models/gemini-3.1-pro-preview                 | []
models/gemini-3.1-pro-preview-customtools     | []
models/gemini-3.1-flash-lite-preview          | []
models/gemini-3.1-flash-lite       

### Transforma chunks em embeddings

In [5]:
# 4. Criar classe que transforma os chuncks em embeddings (vetores numéricos).

print("inicia transformação em embeddings")
#1. Extrair lista de textos dos chunks
textos = [doc.page_content for doc in chunks] 

#2. Gera vetores com API do Gemini em lotes de 100
batch_size=100
all_embeddings= []

for i in range(0, len(textos), batch_size):
    batch = textos[i:i + batch_size]
    response = CLIENT.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=batch
    )
    # Extrai os vetores retornados no lote
    for e in response.embeddings:
        all_embeddings.append(e.values)
    
    print(f" -> Processados {len(all_embeddings)}/{len(textos)} chunks...")

    if len(all_embeddings) < len(textos): #pausa para não atingir erro 429
        time.sleep(60)

print(f"\nTotal de {len(all_embeddings)} vetores gerados com sucesso!")

inicia transformação em embeddings


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0, description: Embed content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 12h5m7.914877762s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per day limit exceeded on Free Tier. See https:/

### Vetores no FAISS

In [8]:
print("inicia criação de FAISS")
textos_embeddings = list(zip(textos, all_embeddings))
metadatas = [doc.metadata for doc in chunks]

vectorstore = FAISS.from_embeddings(
    text_embeddings=textos_embeddings,
    embedding=None,  # vetores já gerados, não reprocessa
    metadatas=metadatas
)

#4. Salvar indice no disco para pesistencia fisica e não precisar gastar API novamente
faiss_index_path = Path("../data/02_faiss_index")
vectorstore.save_local(str(faiss_index_path))

print(f"Índice FAISS salvo com sucesso em: {faiss_index_path.resolve()}")

inicia criação de FAISS


NameError: name 'textos' is not defined

In [10]:
vectorstore = FAISS.load_local(
    folder_path=str(faiss_index_path),
    embeddings=None,  # Não precisa de embedding model pois os vetores já estão prontos
    allow_dangerous_deserialization=True
)

print("✅ Banco FAISS carregado com sucesso!\n")

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


✅ Banco FAISS carregado com sucesso!



### Agente RAG (Retrieval) e formatação de Resposta

In [ ]:
#1. Prompt de Input por parte do usuário
#2. Transformar o prompt em vetor
#3. recuperar vetores no nosso banco vetorial FAISS
#4. Montar contexto estruturado com retorno de metadados
#5. Prompt especializado em análise comparativa com system_instruction e o prompt_final
#6. Gerar resposta comparativa como o modelo

In [ ]:
#1. Prompt de Input por parte do usuário
print("\n1. Prompt de Input por parte do usuário")
#pergunta = input("Digite a pergunta comparativa (ex: Quais as propostas para saúde e como farão?): ")
pergunta = "O que ambos os candidatos abordam sobre os principais comprometedores do orçamento da união: Previdencia, BPC, Saude, e Educação?"
print(f"⚠️ Nenhuma pergunta digitada. Usando padrão: '{pergunta}'")
'''if not pergunta.strip():
    pergunta = "Quais são as principais propostas dos candidatos para a saúde e como eles abordam que isso será feito?"
    print(f"⚠️ Nenhuma pergunta digitada. Usando padrão: '{pergunta}'")'''

#2. Transformar o prompt em vetor
query_response = CLIENT.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=pergunta
)
query_vector = query_response.embeddings[0].values
print("\n2. Prompt transformado em vetor")

#3. recuperar vetores no nosso banco vetorial FAISS k=50
documentos_recuperados = vectorstore.similarity_search_by_vector(query_vector, k=50)
print("\n3. Vetores no banco vetorial FAISS recuperados.")

#4. Montar contexto estruturado com retorno de metadados
contexto_formatado = ""
for i, doc in enumerate(documentos_recuperados):
    candidato = doc.metadata.get("nome_candidato", "N/A")
    partido = doc.metadata.get("sigla_partido", "N/A")
    pagina = doc.metadata.get("page", "N/A")

    contexto_formatado += f"\n--- [TRECHO {i+1}] ---\n"
    contexto_formatado += f"CANDIDATO: {candidato} ({partido}) | PÁGINA: {pagina}\n"
    contexto_formatado += f"CONTEÚDO:\n{doc.page_content}\n"

print("\n4. Contexto estruturado com retorno de metadados estruturado")

#5. Prompt especializado PACE em análise comparativa com system_instruction e o prompt_final
system_instruction = """Você é um economista e analista de políticas públicas sênior, imparcial e altamente crítico, especializado nas Eleições Presidenciais de 2026.

Sua tarefa é realizar uma ANÁLISE CRÍTICA E QUANTITATIVA sobre as propostas dos candidatos com base EXCLUSIVAMENTE no contexto fornecido.

ESTRUTURA OBRIGATÓRIA DA RESPOSTA:

Para CADA candidato identificado no contexto:
1. **Resumo das Propostas & Números Chave:**
   - Liste as propostas identificando os mecanismos de execução ("Como fará").
   - Destaque OBRIGATORIAMENTE todos os dados quantitativos explicitados no texto (valores orçamentários, percentuais, prazos, metas ou métricas). Se o documento não trouxer números, declare: "O plano não apresenta dados quantitativos ou metas numéricas explícitas para este tema."
   - Cite a página de origem [Pág. X].

2. **Análise Crítica de Impactos (Cenário Brasil 2026):**
   - **Impactos Positivos (Oportunidades):** Quais avanços sociais, econômicos ou administrativos a medida pode gerar?
   - **Riscos e Pontos de Atenção (Desafios):** Avalie a viabilidade fiscal, orçamentária e operacional considerando o atual cenário do país (ex: espaço no orçamento, necessidade de aprovação legislativa, complexidade de implementação).

3. **Quadro Comparativo Final (Markdown):**
   Crie uma tabela final consolidando:
   | Candidato (Partido) | Proposta Principal | Dados Quantitativos/Metas | Prós | Contras / Riscos Fiscais e Operacionais |
   
REGRAS RÍGIDAS:
- Mantenha tom neutro, técnico e estritamente focado em políticas públicas.
- Não tome partido; aplique o mesmo rigor crítico a todos os planos analisados."""

prompt_final = f"""CONTEXTO DOS PLANOS DE GOVERNO:
{contexto_formatado}

PERGUNTA DE ANÁLISE:
{pergunta}"""

print("\n5. Prompt especializado em análise comparativa com system_instruction e o prompt_final montado")

#6. Gerar resposta comparativa como o modelo
print("\n6. Gerando resposta comparativa com o modelo...")
resposta = CLIENT.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt_final,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.1 #Rigor analítico com baixa alucinação
    )
)

print("\n" + "=" * 60)
print("📊 RESPOSTA ANALÍTICA E COMPARATIVA")
print("=" * 60 + "\n")
print(resposta.text)



1. Prompt de Input por parte do usuário
⚠️ Nenhuma pergunta digitada. Usando padrão: 'O que ambos os candidatos abordam sobre os principais comprometedores do orçamento da união: Previdencia, BPC, Saude, e Educação?'

2. Prompt transformado em vetor

3. Vetores no banco vetorial FAISS recuperados.

4. Contexto estruturado com retorno de metadados estruturado

5. Prompt especializado em análise comparativa com system_instruction e o prompt_final montado

6. Gerando resposta comparativa com o modelo...

📊 RESPOSTA ANALÍTICA E COMPARATIVA

Com base exclusivamente nos trechos fornecidos dos planos de governo dos candidatos Bolsonaro (PL) e Lula (PT), apresento a análise crítica e quantitativa sobre a abordagem de ambos em relação aos principais comprometedores do orçamento da União (Previdência, BPC, Saúde e Educação).

---

### CANDIDATO: Bolsonaro (PL)

#### 1. Resumo das Propostas & Números Chave:
* **Previdência / INSS:** Proposta de gestão profissional do INSS, implementação de "paco